### ASX200 Similarity Engine & Snapshot Analysis

This notebook contains:
1. Live Snapshot & SIIF portfolio validation. 
2. Simlarity Engine (Historical Z-Score Database, KNN Index & Sanity Check)
3. Forward Distribution Matrix
4. Final Stock Pipeline Visualisation

It uses a live market-state 'snapshot' input to query our historical database and find the most similar historical analogues using Euclidean distance. 


In [ ]:
import yfinance as yf 
import pandas as pd
import numpy as np 
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import warnings
from sklearn.exceptions import DataConversionWarning
warnings.filterwarnings("ignore", category=UserWarning, module="sklearn")

### 1. Livesnap Shot

In [ ]:
#METRIC CALCULATIONS
def get_current_snapshot(ticker: str) -> dict:
    yt = ticker if ticker.endswith(".AX") else ticker + ".AX"
    market_ticker = "^AXJO"
    
    # Download 2y to have enough history for the Z-score means
    raw = yf.download([yt, market_ticker], period="2y", auto_adjust=True, progress=False)
    
    if raw.empty or yt not in raw['Close']:
        raise ValueError(f"Ticker {yt} data missing.")

    prices = raw["Close"]
    volumes = raw["Volume"]
    
    # Daily Returns
    stock_ret = prices[yt].pct_change().dropna()
    market_ret = prices[market_ticker].pct_change().dropna()

    # 1. Momentum Z-Score
    rolling_12m = stock_ret.rolling(252).apply(lambda x: (1 + x).prod() - 1).dropna()
    momentum_zscore = (rolling_12m.iloc[-1] - rolling_12m.mean()) / rolling_12m.std()

    # 2. Volatility Z-Score
    rolling_vol = stock_ret.rolling(30).std() * np.sqrt(252)
    rolling_vol = rolling_vol.dropna()
    vol_zscore = (rolling_vol.iloc[-1] - rolling_vol.mean()) / rolling_vol.std()

    # 3. Idiosyncratic Volatility (Residual from Market Regression)
    aligned = pd.concat([stock_ret, market_ret], axis=1, sort=True).dropna()
    aligned.columns = ['stock', 'market']
    X, y = aligned['market'], aligned['stock']
    
    cov_matrix = np.cov(y, X)
    beta = cov_matrix[0, 1] / cov_matrix[1, 1]
    
    alpha = y.mean() - beta * X.mean()
    residuals = y - (alpha + beta * X)
    idio_vol = residuals.std() * np.sqrt(252)

    # 4. Liquidity Proxy (Average Daily Dollar Volume)
    dollar_volume = (prices[yt] * volumes[yt]).rolling(30).mean().iloc[-1]

    return {
        "ticker": ticker,
        "date": str(prices.index[-1].date()),
        "momentum_z": round(float(momentum_zscore), 4),
        "volatility_z": round(float(vol_zscore), 4),
        "idio_vol": round(float(idio_vol), 4),
        "dollar_vol_30d": round(float(dollar_volume), 2)
    }


1.1. Portfolio Validation

The following code shows the validation for SIIF's current portfolio (comprised of 10 stocks, as opposed to all ASX200) 

In [ ]:
# SIIF CURRENT PORTFOLIO HOLDINGS
portfolio_tickers = ["PLY", "LAU", "COS", "ANG", "VVA", "WTC", "AUB", "TLX", "XYZ", "DUG"]
results = []

print("--- Extracting Live Metrics for SIIF Portfolio ---")
for t in portfolio_tickers:
    try:
        data = get_current_snapshot(t)
        results.append(data)
        print(f"✅ {t:4} | Snapshot captured.")
    except Exception as e:
        print(f"❌ {t:4} | Error: {e}")

# Results converted to table 
portfolio_df = pd.DataFrame(results)
display(portfolio_df)

portfolio_df.to_csv("portfolio_snapshot.csv", index=False)


### 2. Similarity Engine

Building the KNN-based historical analogue matching system.
- 2.1. Historical Z-score database: computes rolling metrics for all 189 tickers 
- 2.2 KNN index: fit nearest neighbour model in Z-score space
- 2.3. Sanity Check: validates the analogues returned 

2.1. Historical Z-score Database

For each ticker in the master dataset, four rolling metrics are computed at each historical date 
- Momentum (12-month returns relative to historical mean)
- Volatility (30-day realised volatility relative to historical mean)
- Idiosyncratic Volatility (Residual Alpha after removing market influence) 
- Liquidity Proxy (30-day average daily dollar volume) 

All Z-scores use an expanding window (as opposed to rolling) to avoid lookahead bias. Each point is only normalised against its own past.

In [ ]:
#HISTORICAL Z-SCORE DATABASE 
def compute_historical_zscores(master_path: str = "ASX200_MASTER_DATASET.csv",
                                market_ticker: str = "^AXJO") -> pd.DataFrame:
    print("Loading master dataset...")
    master = pd.read_csv(master_path, parse_dates=["Date"])
    master = master.sort_values(["ticker", "Date"]).reset_index(drop=True)

    print("Downloading market index...")
    mkt_raw = yf.download(market_ticker, period="5y", auto_adjust=True, progress=False)
    if isinstance(mkt_raw.columns, pd.MultiIndex):
        mkt_raw.columns = mkt_raw.columns.get_level_values(0)
    market_ret = mkt_raw["Close"].pct_change().dropna()
    market_ret.index = market_ret.index.tz_localize(None)

    all_records = []
    tickers = master["ticker"].unique()
    print(f"Computing Z-scores for {len(tickers)} tickers...")

    for ticker in tickers:
        try:
            df = master[master["ticker"] == ticker].copy()
            df = df.set_index("Date").sort_index()

            if len(df) < 300:
                continue

            stock_ret = df["Close"].pct_change()
            aligned = pd.concat([stock_ret, market_ret], axis=1, sort=True).dropna()
            aligned.columns = ["stock", "market"]

            rolling_12m = stock_ret.rolling(252).apply(lambda x: (1 + x).prod() - 1)
            rolling_vol = stock_ret.rolling(30).std() * np.sqrt(252)

            def rolling_idio(idx):
                window = aligned.loc[:idx].tail(252)
                if len(window) < 60:
                    return np.nan
                s, m = window["stock"], window["market"]
                beta = np.cov(s, m)[0, 1] / np.var(m)
                alpha = s.mean() - beta * m.mean()
                residuals = s - (alpha + beta * m)
                return residuals.std() * np.sqrt(252)

            idio_series = pd.Series(
                [rolling_idio(idx) for idx in df.index],
                index=df.index
            )

            dollar_vol = (df["Close"] * df["Volume"]).rolling(30).mean()

            def expanding_zscore(s):
                return (s - s.expanding().mean()) / s.expanding().std()

            records = pd.DataFrame({
                "ticker": ticker,
                "momentum_z": expanding_zscore(rolling_12m),
                "volatility_z": expanding_zscore(rolling_vol),
                "idio_vol_z": expanding_zscore(idio_series),
                "dollar_vol_z": expanding_zscore(np.log1p(dollar_vol)),
            }, index=df.index)

            records = records.dropna()
            all_records.append(records)

        except Exception as e:
            print(f"❌ {ticker}: {e}")

    db = pd.concat(all_records).reset_index().rename(columns={"index": "Date"})
    print(f"\n✅ Z-score database built: {db['ticker'].nunique()} tickers, {len(db):,} snapshots")
    db.to_csv("historical_zscores.csv", index=False)
    return db

The first run of the next code will be slow for all 189 tickers. The output is saved to 'historical_zscores.csv'. 

If you intend to run this notebook multiple times, skip this cell and load directly from the csv.

In [ ]:
historical_db = compute_historical_zscores("ASX200_MASTER_DATASET.csv")

2.2. KNN Index

We are fitting a K-Nearest Neighbours model on the full historical Z-score database using Euclidean distance in Z-score space.

Given a live snapshot of any ASX stock today, the search function returns the K most similar historical observations across all tickers and dates.

Note: self-matches (same ticker) are excluded from returned analogues.
(16/05/26): Liquidity removed as a feature.

In [ ]:
#KNN INDEX BUILDING AND ANALOGUE SEARCH
FEATURE_COLS = ["momentum_z", "volatility_z", "idio_vol_z"]

def build_knn_index(historical_db: pd.DataFrame, k: int = 10):
    db_clean = historical_db[FEATURE_COLS + ["ticker", "Date", "dollar_vol_z"]].dropna().copy()
    scaler = StandardScaler()
    X = scaler.fit_transform(db_clean[FEATURE_COLS])
    knn = NearestNeighbors(n_neighbors=k + 1, metric="euclidean")
    knn.fit(X)
    print(f"✅ KNN index built on {len(db_clean):,} historical snapshots")
    return knn, scaler, db_clean

def find_analogues(snapshot: dict, knn, scaler, db_clean: pd.DataFrame, k: int = 10, vol_tolerance: float = 1.0) -> pd.DataFrame:
    ticker = snapshot["ticker"]
    stock_hist = db_clean[db_clean["ticker"] == ticker]
    
    if not stock_hist.empty:
        target_log_vol = np.log1p(snapshot["dollar_vol_30d"])
        historical_volumes = db_clean[db_clean["ticker"] == ticker]["dollar_vol_z"]
        target_vol_z = (target_log_vol - historical_volumes.mean()) / (historical_volumes.std() + 1e-8)
    else:
        target_vol_z = 0.0

    lower_bound = target_vol_z - vol_tolerance
    upper_bound = target_vol_z + vol_tolerance
    
    liquidity_matched_db = db_clean[
        (db_clean["dollar_vol_z"] >= lower_bound) & 
        (db_clean["dollar_vol_z"] <= upper_bound)].copy()
    
    if len(liquidity_matched_db) < k:
        liquidity_matched_db = db_clean.copy()

    X_subset = scaler.transform(liquidity_matched_db[FEATURE_COLS])
    
    query = np.array([[
        snapshot["momentum_z"],
        snapshot["volatility_z"],
        snapshot.get("idio_vol_z", snapshot["idio_vol"])
    ]])
    query_scaled = scaler.transform(query)
    
    local_knn = NearestNeighbors(n_neighbors=min(k + 20, len(liquidity_matched_db)), metric="euclidean")
    local_knn.fit(X_subset)
    
    distances, indices = local_knn.kneighbors(query_scaled)

    analogues = liquidity_matched_db.iloc[indices[0]].copy()
    analogues["distance"] = distances[0]
    analogues = analogues[analogues["ticker"] != snapshot["ticker"]]
    analogues = analogues.head(k).reset_index(drop=True)
    return analogues

In [ ]:
#Fitting the KNN index on the historical Z-score databse
knn, scaler, db_clean = build_knn_index(historical_db, k=10)

2.3. Sanity Check 

This code runs the similarity engine on a handful of stocks and manually inspects whether the analogues make intuitive sense.

In [ ]:
# SANITY CHECK: FIND ANALOGUES FOR A PORTFOLIO SNAPSHOT
def run_sanity_check(snapshot: dict, knn, scaler, db_clean: pd.DataFrame, k: int = 10):
    print(f"\n{'='*60}")
    print(f"  ANALOGUES FOR: {snapshot['ticker']}  (as of {snapshot['date']})")
    print(f"{'='*60}")
    print(f"  momentum_z  : {snapshot['momentum_z']}")
    print(f"  volatility_z: {snapshot['volatility_z']}")
    print(f"  idio_vol    : {snapshot['idio_vol']}")
    print(f"  dollar_vol  : {snapshot['dollar_vol_30d']:,.0f}")
    print(f"{'-'*60}")

    analogues = find_analogues(snapshot, knn, scaler, db_clean, k, vol_tolerance=1.0)
    print(analogues[["ticker", "Date", "momentum_z", "volatility_z", "distance"]].to_string(index=False))
    return analogues

In [ ]:
#USING SIIF PORTFOLIO SNAPSHOT FOR SANITY CHECK
portfolio_tickers = ["PLY", "LAU", "COS", "ANG", "VVA", "WTC", "AUB", "TLX", "XYZ", "DUG"]

for t in portfolio_tickers:
    try:
        snap = get_current_snapshot(t)
        analogues = find_analogues(snap, knn, scaler, db_clean, k=10, vol_tolerance=1.0)
        print(f"\n{'='*60}")
        print(f"  ANALOGUES FOR: {t}  (as of {snap['date']})")
        print(f"  momentum_z: {snap['momentum_z']}  |  volatility_z: {snap['volatility_z']}  |  idio_vol: {snap['idio_vol']}")
        print(f"{'='*60}")
        display(analogues[["ticker", "Date", "momentum_z", "volatility_z", "distance"]])
    except Exception as e:
        print(f"❌ {t}: {e}")

10/05/2026 Sanity Check Note

Results look sensible across the portfolio: 
- **PLY** (momentum_z: 2.26, high vol): analogues are DNL consecutive days + APA, WOW — stocks in strong upward momentum phases
- **COS** (momentum_z: -1.46, mild vol): analogues are S32, GMG, WES, MIN — large caps in drawdown periods 
- **AUB** (momentum_z: -2.28, low vol): analogues are CHC, BWP, AFI — defensive/financial names in downtrends 
- **DUG** (momentum_z: 1.35, low vol): BHP appearing 5 times — DUG being matched to BHP in mid-2025 momentum phase 

No obviously wrong matches (e.g. mining stocks appearing as analogues for financials). 
Similarity engine returning coherent results. 

### 3. Forward Returns & Signals

Now that we have historical analogues, we turn the similarity engine into a prediction tool.

General logic: if a stock looked like X in the past, and X subsequently did Y...then maybe our stock will also do Y.

This section has four components:
- 3.1. Forward Return Matrix: for each analogue, look up what actually happened after that date
- 3.2. Probability Distribution: aggregate those outcomes into a statistical summary
- 3.3. Entry/Exit Signals**: trigger buy/avoid signals based on the distribution
- 3.4. Fan Chart: visualise all analogue paths overlaid to show the range of outcomes

3.1. Forward Return Matrix

For each analogue returned by the KNN, we look up the actual price history after the analogue date and compute returns at four horizons: - 1 month (21 days)
- 3 months (63 days)
- 6 months (126 days)
- 12 months (252 days).

This gives us a matrix of future trends for every historical match.

In [ ]:
#FOWARD RETURN CALCULATIONS

def compute_analogue_forward_returns(analogues_df, master_path: str = "ASX200_MASTER_DATASET.csv"):
    #each analogue is given a forward return profile based on its actual price history after the analogue date.
    master = pd.read_csv(master_path, parse_dates=["Date"])
    master = master.sort_values(["ticker", "Date"]).reset_index(drop=True)
    
    horizons = {'1M': 21, '3M': 63, '6M': 126, '12M': 252}
    forward_records = []

    for _, row in analogues_df.iterrows():
        ticker = row['ticker']
        analogue_date = pd.to_datetime(row['Date'])
        
        history = master[master["ticker"] == ticker].sort_values("Date").reset_index(drop=True)
        match_idx = history[history["Date"] == analogue_date].index
        
        if match_idx.empty:
            continue
            
        idx = match_idx[0]
        base_price = history.loc[idx, "Close"]
        
        record = {
            "analogue_ticker": ticker,
            "analogue_date": analogue_date.date(),
            "distance": row["distance"]
        }
        
        for name, days in horizons.items():
            future_idx = idx + days
            if future_idx < len(history):
                future_price = history.loc[future_idx, "Close"]
                record[name] = (future_price / base_price) - 1
            else:
                record[name] = np.nan
                
        forward_records.append(record)
        
    return pd.DataFrame(forward_records)

3.2. Probability Distribution

We aggregate the forward return matrix into an empirical probability distribution showing:
- Win Rate: % of analogues that were positive at each horizon
- Median Return (more robust than mean to outliers)
- Min / Max: worst and best case across all analogues


In [ ]:
def compile_distribution(fwd_df: pd.DataFrame) -> pd.DataFrame:
   #returns a summary table of the forward return distribution for each horizon, including win rate, median, and range.
    if fwd_df.empty:
        print("❌ No forward data available.")
        return None
        
    metrics = {}
    print(f"\n{'='*60}")
    print("  PROBABILISTIC OUTPUT DISTRIBUTION")
    print(f"{'='*60}")
    
    for horizon in ['1M', '3M', '6M', '12M']:
        returns = fwd_df[horizon].dropna()
        if returns.empty:
            continue
            
        win_rate = (returns > 0).sum() / len(returns)
        thin_sample = len(returns) < 5
        
        metrics[horizon] = {
            "Analogues (n)": f"{len(returns)} ⚠️" if thin_sample else len(returns),
            "Win Rate (% Up)": f"{win_rate * 100:.1f}%" + (" ⚠️" if thin_sample else ""),
            "Median Return": f"{returns.median() * 100:.2f}%",
            "Min (Worst Case)": f"{returns.min() * 100:.2f}%",
            "Max (Best Case)": f"{returns.max() * 100:.2f}%",
        }
        
    metrics_df = pd.DataFrame(metrics)
    display(metrics_df)
    return metrics_df

3.3. Entry/Exit Signals

The engine evaluates the 3-month distribution against a strict quantitative framework:

- BUY: 3M win rate ≥ 70% AND 3M median ≥ +5%
- SHORT/AVOID: 3M win rate ≤ 30% AND 3M median ≤ -5%
- NO TRADE: all other cases, parameters neutral

If a BUY is triggered:
- Take profit set at the historical 3M median path
- Stop loss floored at -15% per project spec (raw historical min can be extreme and isn't usable for a real fund position)

In [ ]:
#EXNTRY AND EXIT SIGNAL FORMATION
def compile_signals(fwd_df: pd.DataFrame):
    #generates entry/exit signals from the 3M forward return distribution.
    if fwd_df.empty or '3M' not in fwd_df.columns:
        print("❌ No 3M data available for signal generation.")
        return
        
    m3_data = fwd_df['3M'].dropna()
    if m3_data.empty:
        print("❌ Insufficient 3M observations.")
        return
        
    m3_win = (m3_data > 0).sum() / len(m3_data)
    m3_med = m3_data.median()
    m3_min = m3_data.min()
    stop_loss = max(m3_min, -0.15)
    
    print(f"\n{'='*60}")
    print("  ENTRY/EXIT SIGNALS (based on 3M distribution)")
    print(f"{'='*60}")
    print(f"  3M Win Rate  : {m3_win*100:.1f}%")
    print(f"  3M Median    : {m3_med*100:.2f}%")
    print(f"{'-'*60}")
    
    if m3_win >= 0.70 and m3_med >= 0.05:
        print(f"🟢 SIGNAL: BUY ENTRY")
        print(f"💸TAKE PROFIT : Close at +{m3_med*100:.2f}% (median path)")
        print(f"🛑 STOP LOSS   : -{abs(stop_loss)*100:.2f}% from entry")
    elif m3_win <= 0.30 and m3_med <= -0.05:
        print(f"🔴 SIGNAL: SHORT / AVOID")
    else:
        print(f"🟡 SIGNAL: NO TRADE — parameters neutral")

3.4. Fan Chart

This visualises all analogue forward paths overlaid on a single chart.

Grey dashed line = one historical analogue's actual price path after its match date, indexed to 0% at the match date
Bold line = the consensus median path across all analogues. Green if the median outcome was positive, red if negative.

Vertical dotted lines mark the 1M, 3M, 6M and 12M horizon checkpoints.

This is a visual sense-check of whether the distribution is driven by a few outlier paths or whether there's genuine consensus across the analogues.

In [ ]:
def plot_analogue_trajectories(analogues_df: pd.DataFrame, 
                                master_path: str, 
                                target_ticker: str):
    #fan chart showing forward price paths of all historical analogues. Each path is indexed to 0% at the analogue match date.
    master = pd.read_csv(master_path, parse_dates=["Date"])
    master = master.sort_values(["ticker", "Date"]).reset_index(drop=True)
    
    plt.figure(figsize=(11, 5))
    max_days = 252
    all_paths = []
    
    for _, row in analogues_df.iterrows():
        ticker = row['ticker']
        analogue_date = pd.to_datetime(row['Date'])
        
        history = master[master["ticker"] == ticker].sort_values(
            "Date").reset_index(drop=True)
        idx = history[history["Date"] == analogue_date].index
        
        if idx.empty:
            continue
            
        start_idx = idx[0]
        end_idx = min(start_idx + max_days + 1, len(history))
        window_prices = history.loc[start_idx:end_idx - 1, 'Close'].values
        
        if len(window_prices) < 21:
            continue
        
        path_returns = (window_prices / window_prices[0] - 1) * 100
        all_paths.append(pd.Series(path_returns))
        plt.plot(path_returns, color='gray', alpha=0.4, linestyle='--', linewidth=0.8)

    if not all_paths:
        print("❌ Insufficient forward data for plot.")
        return

    paths_df = pd.DataFrame(all_paths).ffill(axis=1)
    median_trajectory = paths_df.median(axis=0)
    
    final_return = median_trajectory.iloc[-1]
    line_color = 'green' if final_return > 0 else 'red'
    
    plt.plot(median_trajectory, color=line_color, linewidth=2.5,
             label=f'Median Path ({final_return:.1f}%)')
    
    # Horizon markers
    for days, label in [(21, '1M'), (63, '3M'), (126, '6M'), (252, '12M')]:
        plt.axvline(days, color='lightgray', linewidth=0.8, linestyle=':')
        plt.text(days + 1, plt.ylim()[0] * 0.9, label, 
                 fontsize=7, color='gray')
    
    plt.axhline(0, color='black', linewidth=0.8)
    plt.title(f"Forward Return Fan Chart — {target_ticker} Analogues", 
              fontsize=11, fontweight='bold')
    plt.xlabel("Trading Days Forward")
    plt.ylabel("Return (%)")
    plt.legend(loc='upper left')
    plt.grid(True, alpha=0.15)
    plt.tight_layout()
    plt.show()

### 4. Full Pipeline: Live Execution ###

The following code executes the full stack into a single execution sequence:
1. Pull live snapshot for the target stock
2. Find K nearest historical analogues via KNN + liquidity filter
3. Deduplicate analogues by ticker (remove temporal clustering)
4. Compute forward returns for each analogue
5. Aggregate into probability distribution and generate signals
6. Plot the fan chart

In [ ]:
#FINAL PIPELINE FUNCTION
def run_full_pipeline(target_ticker: str, 
                      knn, scaler, db_clean, 
                      master_path: str = "ASX200_MASTER_DATASET.csv",
                      k: int = 10,
                      vol_tolerance: float = 1.0):
    """
    Full pipeline: snapshot → analogues → forward returns → signals → chart.
    """
    print(f"\n{'='*60}")
    print(f"  RUNNING SCREENER FOR: {target_ticker}")
    print(f"{'='*60}")
    
    #1. Live snapshot
    snapshot = get_current_snapshot(target_ticker)
    print(f"  Snapshot date  : {snapshot['date']}")
    print(f"  momentum_z     : {snapshot['momentum_z']}")
    print(f"  volatility_z   : {snapshot['volatility_z']}")
    print(f"  idio_vol       : {snapshot['idio_vol']}")
    
    #2. Fnd analogues
    analogues = find_analogues(snapshot, knn, scaler, db_clean, k=20, vol_tolerance=vol_tolerance)
    
    #3. Deduplicate temporal clustering
    analogues = analogues.sort_values("distance").drop_duplicates(subset="ticker", keep="first").reset_index(drop=True)
    print(f"\n  Analogues found: {len(analogues)} (after deduplication)")
    
    #4. Forward returns
    fwd_returns = compute_analogue_forward_returns(analogues, master_path)
    
    #5. Distribution and signals
    metrics = compile_distribution(fwd_returns)
    compile_signals(fwd_returns)
    
    #6. Fan chart
    plot_analogue_trajectories(analogues, master_path, target_ticker)
    
    return snapshot, analogues, fwd_returns, metrics

In [ ]:
#Final SIIF PORTFOLIO RUN
portfolio_tickers = ["PLY", "LAU", "COS", "ANG", "VVA", "WTC", "AUB", "TLX", "XYZ", "DUG"]

for t in portfolio_tickers:
    try:
        run_full_pipeline(t, knn, scaler, db_clean, k=20)
    except Exception as e:
        print(f"❌ {t}: {e}")

### 15/05/2026 Final Live Output Note: ###

| Ticker | Signal    | Win Rate (3M) | Median (3M) | Take Profit | Stop Loss |
|--------|-----------|---------------|-------------|-------------|-----------|
| PLY    | NO TRADE  | 46.2%         | -1.62%      | —           | —         |
| LAU    | NO TRADE  | 53.8%         | +5.61%      | —           | —         |
| COS    | NO TRADE  | 50.0%         | +2.51%      | —           | —         |
| ANG    | NO TRADE  | 54.5%         | +0.59%      | —           | —         |
| VVA    | NO TRADE  | 44.4%         | -1.63%      | —           | —         |
| WTC    | NO TRADE  | 54.5%         | +3.04%      | —           | —         |
| AUB    | 🟢 BUY   | 81.8%         | +5.00%      | +5.00%      | -8.37%    |
| TLX    | NO TRADE  | 60.0%         | +1.38%      | —           | —         |
| XYZ    | NO TRADE  | 36.4%         | -4.05%      | —           | —         |
| DUG    | NO TRADE  | 36.4%         | -5.79%      | —           | —         |

Interesting things to flag: 
- **LAU** narrowly missed BUY on win rate (53.8% vs 70% threshold) despite strong median (+5.61%)
- **XYZ** and DUG both bearish but just above SHORT threshold on win rate...worth monitoring
- **AUB** take profit (+5.00%) vs stop loss (-8.37%) is negative risk/reward...consider using 75th percentile return as take profit instead of median
- **AND** logic on signal criteria is strict, discuss whether OR makes more sense